# Time Profiling: encontrando cuellos de botella

Cuando un programa va lento, la pregunta no es "¿cómo lo hago más rápido?", sino una
anterior y más importante:

> **¿Qué parte de mi programa está tardando realmente?**

El error más común al empezar es **optimizar a ciegas**: reescribir un trozo de código
que *creemos* lento, cuando muchas veces el verdadero cuello de botella está en otro
sitio que ni sospechábamos. Por eso la regla de oro de este vídeo es:

> 🎯 **Primero mide, después optimiza.**

Vamos a aprender las herramientas para medir, de la más simple a la más completa:

- **`time.perf_counter()`**: cronometrar un trozo de código a mano.
- **`timeit`**: medir fragmentos pequeños con rigor (varias ejecuciones).
- **`cProfile`**: el núcleo del vídeo, para ver **en qué función** se va el tiempo.
- **SnakeViz**: visualizar ese perfil de forma gráfica.
- Un **caso práctico** comparando alternativas reales.

> 📚 Documentación de referencia:
> **[`timeit`](https://docs.python.org/es/3/library/timeit.html)** y
> **[`profile`/`cProfile`](https://docs.python.org/es/3/library/profile.html)**.

---

# Parte 1: Timing rápido con `time.perf_counter()`

## 1. Cronometrar un trozo de código

La forma más sencilla de medir cuánto tarda algo es tomar el tiempo **antes** y
**después** y restar. Para eso usamos `time.perf_counter()`:

In [ ]:
from time import perf_counter


def procesar_datos():
    total = 0
    for i in range(1_000_000):
        total += i
    return total


start = perf_counter()
procesar_datos()
end = perf_counter()

print(f"Tardó {end - start:.4f} segundos")

### ¿Por qué `perf_counter()` y no `time.time()`?

Podrías pensar en usar `time.time()`, pero **no es lo adecuado para medir duraciones**:

| | `time.time()` | `time.perf_counter()` |
|---|---------------|------------------------|
| Qué mide | La hora "de pared" (fecha/hora real) | Un contador pensado para medir intervalos |
| Resolución | Más baja | La **más alta** disponible en el sistema |
| ¿Puede "saltar"? | Sí: si el reloj del sistema se ajusta (NTP, cambio de hora) puede ir hacia atrás | No: es **monótono**, siempre avanza |

Para **medir cuánto tarda algo**, siempre `perf_counter()`. `time.time()` es para saber
*qué hora es*, no para cronometrar.

> 💡 Esta técnica está bien para **medidas rápidas** y *benchmarks* improvisados, pero
> tiene un problema: una sola ejecución es muy sensible al "ruido" (otros procesos del
> ordenador, etc.). Para fragmentos pequeños, hay algo mejor.

---

# Parte 2: Medir con rigor usando `timeit`

## 2. Muchas ejecuciones, resultados fiables

El módulo `timeit` está hecho específicamente para medir trozos pequeños de código.
Su truco es **ejecutarlos muchas veces** y así diluir el ruido de una medición suelta.

In [1]:
import timeit

# Ejecuta 'sum(range(1000))' 10 000 veces y devuelve el tiempo TOTAL
total = timeit.timeit("sum(range(1000))", number=10000)
print(f"Total para 10 000 ejecuciones: {total:.4f} s")
print(f"Tiempo medio por ejecución:    {total / 10000 * 1e6:.2f} µs")

Total para 10 000 ejecuciones: 0.1388 s
Tiempo medio por ejecución:    13.88 µs


Lo interesante de `timeit` es para **comparar dos formas** de hacer lo mismo y ver
cuál es más rápida. Por ejemplo, ¿es mejor `lista.append(x)` o `lista += [x]`?

Fíjate en el parámetro `setup`: es código que se ejecuta **una vez** para preparar el
terreno (aquí, crear la lista), y que **no** se cuenta en el tiempo medido.

In [2]:
import timeit

append_t = timeit.timeit("lista.append(1)", setup="lista = []", number=1_000_000)
concat_t = timeit.timeit("lista += [1]", setup="lista = []", number=1_000_000)

print(f"lista.append(1) : {append_t:.4f} s")
print(f"lista += [1]    : {concat_t:.4f} s")

lista.append(1) : 0.0119 s
lista += [1]    : 0.0545 s


### Conceptos clave de una medición seria

- **Múltiples ejecuciones**: medir una vez no dice casi nada; medir miles de veces sí.
- **Warm-up (calentamiento)**: las primeras ejecuciones suelen ser más lentas (cachés,
  inicializaciones...). Repetir muchas veces hace que ese efecto inicial se diluya.
- **Promedios... o mejor, el mínimo**: para comparar, suele usarse el **tiempo mínimo**
  de varias tandas, porque el ruido externo solo puede *sumar* tiempo, nunca restarlo.

Para eso, `timeit.repeat` ejecuta varias **tandas** y nos quedamos con la mejor:

In [3]:
import timeit

tandas = timeit.repeat("sum(range(1000))", number=10000, repeat=5)
print("Tiempos de cada tanda:", [f"{t:.4f}" for t in tandas])
print(f"Mejor tiempo (el más representativo): {min(tandas):.4f} s")

Tiempos de cada tanda: ['0.2303', '0.2271', '0.2282', '0.2274', '0.2257']
Mejor tiempo (el más representativo): 0.2257 s


---

# Parte 3: `cProfile`, el núcleo del profiling

## 3. ¿Dónde se va el tiempo de TODO el programa?

`perf_counter` y `timeit` miden trozos que tú eliges. Pero, ¿y si no sabes **qué**
medir? Aquí entra `cProfile`: ejecuta tu programa entero y te dice, **función por
función**, cuánto tiempo se consume en cada una. Es la herramienta para **encontrar el
cuello de botella** sin adivinar.

Montemos un ejemplo realista con la biblioteca del curso: una función `buscar` que,
por un mal diseño, **normaliza el texto demasiadas veces**.

In [4]:
def normalizar(texto):
    # simulamos una operación de normalización "cara"
    resultado = texto
    for _ in range(500):
        resultado = texto.strip().lower()
    return resultado


LIBROS = [f"  Libro {i}  " for i in range(300)]


def buscar(consulta):
    objetivo = normalizar(consulta)
    encontrados = []
    for titulo in LIBROS:
        if normalizar(titulo) == objetivo:  # ⚠️ normaliza CADA título en CADA búsqueda
            encontrados.append(titulo.strip())
    return encontrados


def main():
    for _ in range(20):
        buscar("  Libro 150  ")


print("Resultado:", buscar("  Libro 150  "))

Resultado: ['Libro 150']


Ahora lanzamos el *profiler* sobre `main()`. Ordenamos por `tottime` para ver primero
las funciones que más tiempo consumen **en sí mismas**:

In [5]:
import cProfile

cProfile.run("main()", sort="tottime")

         6026700 function calls (6026691 primitive calls) in 2.372 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     6020    1.188    0.000    2.355    0.000 2943246465.py:1(normalizar)
  3010020    0.590    0.000    0.590    0.000 {method 'strip' of 'str' objects}
  3010000    0.578    0.000    0.578    0.000 {method 'lower' of 'str' objects}
        1    0.011    0.011    0.020    0.020 history.py:100(only_when_enabled)
       20    0.002    0.000    2.316    0.116 2943246465.py:12(buscar)
      2/1    0.000    0.000    2.360    2.360 <string>:1(<module>)
       14    0.000    0.000    0.000    0.000 socket.py:623(send)
        2    0.000    0.000    0.011    0.005 zmqstream.py:546(_run_callback)
        1    0.000    0.000    0.000    0.000 {method 'execute' of 'sqlite3.Connection' objects}
        4    0.000    0.000    0.000    0.000 {method 'get' of 'dict' objects}
        1    0.000    0.000    0.000    0.000 {met

## 4. Leer la salida de `cProfile`

Las columnas que de verdad importan son tres:

| Columna | Qué significa |
|---------|----------------|
| **ncalls** | Número de **llamadas** a la función |
| **tottime** | Tiempo gastado **dentro** de la función, *sin* contar las subfunciones que llama |
| **cumtime** | Tiempo **acumulado**: el de la función *más* el de todo lo que llama por dentro |

La clave es la diferencia entre `tottime` y `cumtime`:

- Un `cumtime` alto en `main` o `buscar` es normal: incluye todo lo que llaman.
- El **culpable real** suele ser quien tiene un `tottime` alto. En nuestro caso es
  `normalizar`: se la llama **miles de veces** (mira su `ncalls`) y ahí está casi todo
  el tiempo.

El perfil nos revela la estructura del problema:

```
buscar()
└── normalizar()   ← aquí se va la mayor parte del tiempo
```

> 💡 Con este dato, la optimización es evidente y **dirigida**: normalizar cada título
> una sola vez (por ejemplo, guardando los títulos ya normalizados) en lugar de
> hacerlo en cada búsqueda. Sin el *profiler*, podríamos haber perdido el tiempo
> optimizando el bucle de `main`, que no era el problema.

---

# Parte 4: Visualizar el perfil con SnakeViz

## 5. De la tabla al gráfico

La tabla de `cProfile` es muy útil, pero cuesta leerla cuando hay muchas funciones.
**SnakeViz** es una herramienta que convierte ese perfil en un **gráfico interactivo**
en el navegador.

Primero la instalamos como dependencia de desarrollo (como vimos con otras
herramientas del curso):

```bash
uv add --dev snakeviz
```

Luego guardamos el perfil en un archivo (`-o profile.prof`) y lo abrimos con SnakeViz:

```bash
python -m cProfile -o profile.prof programa.py
snakeviz profile.prof
```

Se abrirá una página web con dos vistas principales:

- **Flame graph (gráfico de llamas / *icicle*)**: cada barra es una función y su
  **anchura** es proporcional al tiempo que consume. De un vistazo ves qué función es
  "la más ancha": ese es tu cuello de botella.
- **Call graph (tabla de llamadas)**: la misma información de `cProfile` pero ordenable
  e interactiva, para explorar quién llama a quién.

> 💡 SnakeViz no añade información nueva sobre `cProfile`: la **muestra mejor**. Para
> programas grandes, ver el *flame graph* hace que el cuello de botella salte a la
> vista al instante.

---

# Parte 5: Caso práctico — medir antes de elegir

## 6. La estructura de datos correcta gana a cualquier microoptimización

Vamos a aplicar lo aprendido para tomar una decisión real con datos, no con
intuiciones. Compararemos dos cosas que se hacen constantemente.

**Comprobar si un elemento está en una `list` vs en un `set`:**

In [ ]:
import timeit

# Buscamos un elemento que está al final (el peor caso para la lista)
t_lista = timeit.timeit("9999 in datos", setup="datos = list(range(10000))", number=10000)
t_set = timeit.timeit("9999 in datos", setup="datos = set(range(10000))", number=10000)

print(f"Buscar en list: {t_lista:.4f} s")
print(f"Buscar en set : {t_set:.4f} s")
print(f"El set es ~{t_lista / t_set:.0f} veces más rápido para esta búsqueda")

La diferencia es enorme: una `list` tiene que recorrer elemento a elemento (`in` es
lento), mientras que un `set` encuentra el elemento casi instantáneamente. Si tu
programa hace **muchas búsquedas de pertenencia**, elegir un `set` puede ser la
optimización más rentable, y solo lo sabemos **porque lo hemos medido**.

**Construir una lista: bucle `for` con `append` vs *list comprehension*:**

In [ ]:
import timeit

codigo_for = """
cuadrados = []
for i in range(1000):
    cuadrados.append(i * i)
"""

t_for = timeit.timeit(codigo_for, number=10000)
t_comp = timeit.timeit("[i * i for i in range(1000)]", number=10000)

print(f"for + append      : {t_for:.4f} s")
print(f"list comprehension: {t_comp:.4f} s")

La *list comprehension* suele ser algo más rápida (y más legible) que el bucle con
`append`. Pero ojo: esta es una **microoptimización**. Comparada con elegir bien la
estructura de datos (`list` vs `set`), el impacto es pequeño. Por eso primero se
perfila (`cProfile`) para encontrar el cuello de botella **real**, y solo entonces se
afina ese punto concreto.

---

## 7. Resumen

La idea central del vídeo cabe en una frase: **primero mide, después optimiza**.
Optimizar a ciegas es perder el tiempo en el sitio equivocado. Las herramientas, de
menos a más potente:

1. **`time.perf_counter()`**: cronometrar a mano un trozo de código. Úsalo en vez de
   `time.time()`, porque es monótono y de alta resolución.
2. **`timeit`**: medir fragmentos pequeños con rigor, ejecutándolos muchas veces;
   ideal para **comparar** dos alternativas. Quédate con el **mínimo** de varias tandas.
3. **`cProfile`**: el núcleo. Perfila el programa entero y te dice en qué función se va
   el tiempo (`ncalls`, `tottime`, `cumtime`). Sirve para **encontrar el cuello de
   botella**.
4. **SnakeViz**: visualiza ese perfil como un *flame graph* interactivo.
5. **Caso práctico**: medir te deja **elegir con datos** (p. ej. `set` en vez de `list`
   para búsquedas).

| Herramienta | Para qué | Granularidad |
|-------------|----------|--------------|
| `time.perf_counter()` | Cronometrar un bloque concreto | Un trozo que tú eliges |
| `timeit` | Comparar fragmentos pequeños con rigor | Una expresión / sentencia |
| `cProfile` | Encontrar el cuello de botella del programa | Todas las funciones |
| SnakeViz | Ver el perfil de forma gráfica | Visualización del `cProfile` |